##1. Importing required libraries

In [2]:
pip install langchain langchain-openai langchain-groq --quiet

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

In [4]:
from google.colab import userdata
groq_api_key = userdata.get('GROQ_API_KEY')

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.2,
    api_key=groq_api_key
)

##2. Zero shot prompting

In [5]:
zero_shot_prompt = ChatPromptTemplate.from_messages([
    ("human", "Explain what a Python dictionary is and give an example.")
])

chain = zero_shot_prompt | llm
print(chain.invoke({}).content)

**What a Python dictionary is**

A *dictionary* (often called a **dict**) is one of Python’s built‑in data structures. It stores a collection of **key‑value pairs** where:

| Feature | Description |
|---------|--------------|
| **Key** | Must be *hashable* (e.g., strings, numbers, tuples of immutable objects). Keys are unique – adding a duplicate key overwrites the previous value. |
| **Value** | Can be any Python object (including other dictionaries, lists, functions, etc.). |
| **Mutable** | You can add, remove, or change entries after the dictionary is created. |
| **Lookup speed** | Average‑case O(1) time for retrieving a value by its key, because a dict is implemented as a hash table. |
| **Order** | Since Python 3.7 the insertion order is preserved (iteration yields items in the order they were added). |

In everyday language a dictionary is like a real‑world lookup table: you give it a *key* (the word you’re looking up) and it returns the *value* (the definition).

---

**Basic 

##3. One shot prompting



In [6]:
one_shot_prompt = ChatPromptTemplate.from_messages([
    ("human", """
Example:
Q: What is a list in Python?
A: A list is an ordered, mutable collection of elements.

Now answer:
Q: What is a tuple in Python?
""")
])

chain = one_shot_prompt | llm
print(chain.invoke({}).content)

A tuple in Python is an ordered, **immutable** collection of elements. It can hold items of any type, is defined using parentheses (e.g., `my_tuple = (1, "a", 3.14)`) or without them (`my_tuple = 1, "a", 3.14`), and its contents cannot be changed after creation.


##3. Few shot prompting

In [7]:
few_shot_prompt = ChatPromptTemplate.from_messages([
    ("human", """
Q: What is a stack?
A: A linear data structure that follows LIFO.

Q: What is a queue?
A: A linear data structure that follows FIFO.

Q: What is a linked list?
""")
])

chain = few_shot_prompt | llm
print(chain.invoke({}).content)

**Q: What is a linked list?**  
**A:** A linked list is a linear data structure composed of a sequence of *nodes*, where each node stores a data element and one or more references (pointers) to other nodes. In a singly linked list each node points to the next node in the sequence; in a doubly linked list each node points both to the next and the previous node. This pointer‑based organization allows for efficient insertion and deletion of elements at arbitrary positions (typically O(1) time when the node is known), without the need to shift other elements as in an array. However, accessing an element by index requires traversing the list from the head, giving O(n) time for random access.


##4. RGC FRAMEWORK (Role–Goal–Context)

In [8]:
rgc_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a computer science instructor."),
    ("human", """
Role:
You are acting as a teacher.

Goal:
Explain binary search.

Context:
- Audience: Beginner
- Use simple examples
- Avoid heavy math
""")
])

chain = rgc_prompt | llm
print(chain.invoke({}).content)

## Binary Search – A Friendly Introduction  

### 1. What Is Binary Search?

Imagine you have a **sorted** list of items (numbers, words, names …) and you want to know whether a particular item is in that list.  
Instead of looking at every single element one‑by‑one (that’s called *linear search*), binary search repeatedly cuts the list in half and discards the half that cannot contain the item.  

Because we keep halving the search space, we find the answer **very quickly** – even for huge lists.

> **Key requirement:** The list must be **sorted** from low to high (or alphabetically). If it isn’t sorted, binary search won’t work.

---

### 2. The Core Idea – “Guess the Middle”

Think of the list as a book’s index:

```
[ 2, 5, 8, 12, 16, 23, 38, 41, 56, 72 ]
```

You’re looking for the number **23**.

1. **Look at the middle element**  
   - The middle of 10 items is the 5th (or 6th) element.  
   - Here the middle value is **16**.

2. **Compare** the middle value with the target (23)

##5. Chain of thought


In [9]:
cot_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a logical problem solver.
Think step-by-step internally.
Do NOT reveal raw chain-of-thought.
Provide a short structured explanation.
"""),
    ("human", """
If a function runs in O(n) time and is called n times,
what is the overall time complexity?
""")
])

chain = cot_prompt | llm
print(chain.invoke({}).content)

**Answer:** \(O(n^2)\)

**Reasoning**

1. One execution of the function costs at most \(c \cdot n\) steps for some constant \(c\) (by the definition of \(O(n)\)).
2. The function is invoked \(n\) times, so the total work is at most  
   \(n \times (c \cdot n) = c \cdot n^2\).
3. Ignoring constant factors, the overall running time grows proportionally to \(n^2\).

Hence the combined complexity is \(O(n^2)\).


##6. Tabular format prompting

In [11]:
table_prompt = ChatPromptTemplate.from_messages([
    ("human", """
Compare the following data structures in a table:
- Array
- Linked List
- Stack

Columns:
- Structure
- Access Time
- Insertion Time
- Use Case
""")
])

chain = table_prompt | llm
print(chain.invoke({}).content)

**Comparison of Array, Linked List, and Stack**

| Structure | Access Time | Insertion Time* | Typical Use Case |
|-----------|-------------|-----------------|------------------|
| **Array** (static or dynamic) | **O(1)** – direct indexing by position | **O(n)** in the worst case (shifting elements) – **O(1)** amortized for appending to a dynamic array (e.g., `ArrayList`, `vector`) | Storing a fixed‑size collection where fast random access is required (e.g., lookup tables, matrix representations, buffers) |
| **Linked List** (singly or doubly) | **O(n)** – must traverse from the head to reach an element | **O(1)** if you already have a reference to the insertion point (e.g., at the head or after a known node); otherwise **O(n)** to locate the spot | Scenarios with frequent insertions/deletions in the middle of a sequence and where memory reallocation is undesirable (e.g., implementing queues, adjacency lists in graphs) |
| **Stack** (LIFO) – usually implemented with an array or linked 

##7. Fill in the blank prompting

In [12]:
fill_blank_prompt = ChatPromptTemplate.from_messages([
    ("human", """
Fill in the blanks:

In Python, a ______ is used to store key-value pairs.
It is ______ (mutable / immutable).
""")
])

chain = fill_blank_prompt | llm
print(chain.invoke({}).content)

In Python, a **dictionary** is used to store key‑value pairs.  
It is **mutable**.


##8. Combined Practise Prompting

In [13]:
combined_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a senior Coding AI.
Reason internally step-by-step.
Do not expose raw chain-of-thought.
"""),
    ("human", """
Role:
You are a DSA instructor.

Goal:
Explain time complexity classes.

Context:
- Audience: Beginners
- Output format: Table

Instructions:
- Keep explanations short
- Use examples
""")
])

chain = combined_prompt | llm
print(chain.invoke({}).content)

**Time‑Complexity Cheat‑Sheet**

| Complexity | What it means (in plain words) | Typical example (simple algorithm) |
|------------|--------------------------------|-------------------------------------|
| **O(1)**   | Runs in constant time – the work does **not** grow with input size. | Accessing an array element `a[i]`. |
| **O(log n)** | Work grows logarithmically; each step cuts the problem size roughly in half. | Binary search in a sorted array. |
| **O(n)**   | Work grows linearly with the number of items. | Scanning a list to find the maximum. |
| **O(n log n)** | Linear work *plus* a logarithmic factor; common for “divide‑and‑conquer” sorts. | Merge sort or heap sort. |
| **O(n²)**  | Quadratic growth – often from two nested loops over the same data. | Naïve bubble sort; checking all pairs in a list. |
| **O(2ⁿ)**  | Exponential growth – the work doubles for each extra element. | Generating all subsets of a set (power set). |
| **O(n!)**  | Factorial growth – the work multiplie